# Swin Transformer V2 + Graph Attention Fusion

Swin-style transformer image backbone with graph attention over metadata nodes.

This notebook is optimized for Kaggle GPU execution and uses the shared ISIC split structure with leakage-safe image/metadata alignment.

In [ ]:

import os
import re
import math
import json
import copy
import random
import warnings
from dataclasses import dataclass
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image, ImageFilter, ImageOps, ImageEnhance
from IPython.display import display

import cv2
from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torch.cuda.amp import autocast, GradScaler

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    confusion_matrix,
    roc_curve,
    precision_recall_curve,
    brier_score_loss,
)

try:
    from torchvision import transforms
except Exception as e:
    raise RuntimeError(
        "torchvision is required for this notebook. On Kaggle it is usually available. "
        f"Import failed with: {e}"
    )

warnings.filterwarnings("ignore")
plt.style.use("default")
pd.set_option("display.max_columns", 200)

# =========================================================
# CONFIG
# =========================================================
@dataclass
class CFG:
    IMG_SIZE = 384
    MEAN = (0.485, 0.456, 0.406)
    STD = (0.229, 0.224, 0.225)
    D = 128
    NUM_HEADS = 8
    FF_DIM = 256
    NUM_CA_LAYERS = 2
    DROPOUT_TF = 0.15
    TAB_DEPTH = 2
    TAB_HEADS = 4
    FREEZE_BLOCKS = 4
    SEED = 42
    EPOCHS = 3
    BATCH_SIZE = 16
    GRAD_ACCUM = 2
    NUM_WORKERS = 2
    PATIENCE = 10
    LABEL_SMOOTH = 0.10
    FOCAL_ALPHA = 0.25
    FOCAL_GAMMA = 2.0
    LR_BACKBONE = 2e-5
    LR_HEAD = 2e-4
    WEIGHT_DECAY = 1e-4
    WARMUP_EPOCHS = 3
    EMA_ALPHA = 0.3
    CUTMIX_PROB = 0.00   # disabled by default for multimodal safety
    MIXUP_PROB = 0.00    # disabled by default for multimodal safety
    TTA_STEPS = 5
    N_FOLDS = 5

    XAI_N_SAMPLES = 5
    XAI_IG_STEPS = 20
    LIME_N_SAMPLES = 500
    LIME_N_SEGMENTS = 60
    SHAP_BACKGROUND = 30
    SHAP_NSAMPLES = 50
    SHAP_EXPLAIN = 15
    PDP_N_FEATURES = 6
    PDP_GRID_POINTS = 50

CFG = CFG()

MODEL_NAME = "Swin Transformer V2 + Graph Attention Fusion"
print(f"Model: {MODEL_NAME}")

# =========================================================
# SEEDS / DEVICE
# =========================================================
def seed_everything(seed: int = 42) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    torch.backends.cudnn.deterministic = False
    torch.backends.cudnn.benchmark = True

seed_everything(CFG.SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

# =========================================================
# DATA DISCOVERY
# =========================================================
def find_first_existing(paths):
    for p in paths:
        if p and os.path.exists(p):
            return p
    return None

def walk_for_csvs(base_dirs=("/kaggle/input", "/kaggle/working")):
    hits = []
    for base in base_dirs:
        if not os.path.exists(base):
            continue
        for root, _, files in os.walk(base):
            low = {f.lower() for f in files}
            if {"train.csv", "val.csv", "test.csv"}.issubset(low):
                hits.append(root)
    hits = sorted(set(hits), key=lambda x: len(x))
    return hits

def extract_if_needed():
    # Optional extraction for zip-based submissions
    base_dirs = ["/kaggle/input", "/kaggle/working"]
    for base in base_dirs:
        if not os.path.exists(base):
            continue
        for root, _, files in os.walk(base):
            low = {f.lower(): f for f in files}
            if "train.zip" in low and "val.zip" in low and "test.zip" in low:
                out_root = "/kaggle/working/extracted_split"
                os.makedirs(out_root, exist_ok=True)
                for split in ["train", "val", "test"]:
                    zpath = os.path.join(root, low[f"{split}.zip"])
                    target = os.path.join(out_root, split)
                    if not os.path.exists(target):
                        import zipfile
                        with zipfile.ZipFile(zpath, "r") as zf:
                            zf.extractall(out_root)
                return out_root
    return None

split_roots = walk_for_csvs()
if split_roots:
    DATA_ROOT = split_roots[0]
else:
    extracted = extract_if_needed()
    if extracted is None:
        raise FileNotFoundError(
            "Could not auto-detect the split root. Expected train.csv/val.csv/test.csv "
            "or train.zip/val.zip/test.zip somewhere under /kaggle/input or /kaggle/working."
        )
    DATA_ROOT = extracted

print("DATA_ROOT:", DATA_ROOT)

def locate_split_csv(split):
    candidates = [
        os.path.join(DATA_ROOT, split, f"{split}.csv"),
        os.path.join(DATA_ROOT, f"{split}.csv"),
        os.path.join(DATA_ROOT, split, "labels.csv"),
        os.path.join(DATA_ROOT, split, "metadata.csv"),
    ]
    found = find_first_existing(candidates)
    if found:
        return found
    # fallback recursive search
    for root, _, files in os.walk(DATA_ROOT):
        for f in files:
            if f.lower() == f"{split}.csv":
                return os.path.join(root, f)
    raise FileNotFoundError(f"Could not find {split}.csv under {DATA_ROOT}")

TRAIN_CSV = locate_split_csv("train")
VAL_CSV   = locate_split_csv("val")
TEST_CSV  = locate_split_csv("test")

def infer_image_dirs(csv_path):
    root = os.path.dirname(csv_path)
    candidates = [
        os.path.join(root, "images"),
        os.path.join(root, "image"),
        root,
        os.path.join(DATA_ROOT, "images"),
        os.path.join(DATA_ROOT, "train", "images"),
        os.path.join(DATA_ROOT, "val", "images"),
        os.path.join(DATA_ROOT, "test", "images"),
    ]
    out = []
    for c in candidates:
        if os.path.exists(c) and os.path.isdir(c):
            out.append(c)
    # remove duplicates preserve order
    uniq = []
    for c in out:
        if c not in uniq:
            uniq.append(c)
    return uniq

# =========================================================
# COLUMN INFERENCE
# =========================================================
def pick_col(df, candidates, required=True, fallback=None):
    low = {c.lower(): c for c in df.columns}
    for cand in candidates:
        if cand.lower() in low:
            return low[cand.lower()]
    if fallback is not None:
        return fallback
    if required:
        raise KeyError(f"Could not infer column from candidates: {candidates}. Found columns: {df.columns.tolist()}")
    return None

def canonical_image_name(name):
    if pd.isna(name):
        return None
    s = str(name)
    base = os.path.basename(s)
    if re.search(r'\.(jpg|jpeg|png|bmp|webp)$', base, flags=re.I):
        return base
    nums = re.findall(r'(\d{7})', s)
    if nums:
        return f"ISIC_{nums[-1]}.jpg"
    return base

def encode_binary_label(v):
    if pd.isna(v):
        return 0
    if isinstance(v, (int, float, np.integer, np.floating)):
        return int(v)
    s = str(v).strip().lower()
    pos = {"1", "true", "yes", "y", "melanoma", "mel", "malignant", "cancer", "positive"}
    neg = {"0", "false", "no", "n", "non-melanoma", "benign", "nevus", "negative", "oth", "other"}
    if s in pos:
        return 1
    if s in neg:
        return 0
    # fallback heuristics
    if "mel" in s and "non" not in s:
        return 1
    return 0

def humanize_label_cols(df):
    image_col = pick_col(df, ["image_fixed", "image", "filename", "file_name", "isic_id"], required=True)
    label_col = pick_col(df, ["target", "label", "class", "melanoma", "diagnosis", "is_melanoma"], required=True)

    num_candidates = ["age", "age_approx", "patient_age", "age_years", "age_year"]
    cat_candidates = ["sex", "gender", "anatom_site_general", "anatomical_site", "site", "body_site", "lesion_type", "diagnosis_confirm_type"]

    num_cols = [c for c in num_candidates if c in df.columns]
    cat_cols = [c for c in cat_candidates if c in df.columns]

    # also include any other low-cardinality object columns except image/label
    for c in df.columns:
        if c in {image_col, label_col} or c in num_cols or c in cat_cols:
            continue
        if df[c].dtype == object and df[c].nunique(dropna=True) <= 25:
            cat_cols.append(c)

    # keep only unique and existing
    num_cols = [c for i, c in enumerate(num_cols) if c in df.columns and c not in num_cols[:i]]
    cat_cols = [c for i, c in enumerate(cat_cols) if c in df.columns and c not in cat_cols[:i]]

    return image_col, label_col, num_cols, cat_cols

# =========================================================
# IMAGE PREPROCESSING
# =========================================================
def simple_hair_removal_pil(img: Image.Image) -> Image.Image:
    arr = np.array(img.convert("RGB"))
    gray = cv2.cvtColor(arr, cv2.COLOR_RGB2GRAY)
    kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (17, 17))
    blackhat = cv2.morphologyEx(gray, cv2.MORPH_BLACKHAT, kernel)
    _, mask = cv2.threshold(blackhat, 10, 255, cv2.THRESH_BINARY)
    mask = cv2.dilate(mask, np.ones((3, 3), np.uint8), iterations=1)
    out = cv2.inpaint(arr, mask, 1, cv2.INPAINT_TELEA)
    return Image.fromarray(out)

def shades_of_gray_pil(img: Image.Image, power: int = 6) -> Image.Image:
    arr = np.asarray(img.convert("RGB")).astype(np.float32)
    arr = np.maximum(arr, 1.0)
    rgb_power = np.power(arr, power)
    norm = np.power(np.mean(rgb_power, axis=(0, 1)), 1.0 / power)
    scale = np.mean(norm) / (norm + 1e-6)
    arr = np.clip(arr * scale.reshape(1, 1, 3), 0, 255).astype(np.uint8)
    return Image.fromarray(arr)

def rough_lesion_crop(img: Image.Image) -> Image.Image:
    arr = np.array(img.convert("RGB"))
    gray = cv2.cvtColor(arr, cv2.COLOR_RGB2GRAY)
    gray = cv2.GaussianBlur(gray, (5, 5), 0)
    # Choose inverse threshold if lesion is darker than background
    _, th1 = cv2.threshold(gray, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    th2 = 255 - th1
    masks = [th1, th2]
    best = None
    best_area = 0
    for mask in masks:
        cnts, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        if not cnts:
            continue
        cnt = max(cnts, key=cv2.contourArea)
        area = cv2.contourArea(cnt)
        if area > best_area:
            best_area = area
            best = cnt
    if best is None or best_area < 500:
        return img
    x, y, w, h = cv2.boundingRect(best)
    pad = int(0.15 * max(w, h))
    x1 = max(x - pad, 0)
    y1 = max(y - pad, 0)
    x2 = min(x + w + pad, arr.shape[1])
    y2 = min(y + h + pad, arr.shape[0])
    cropped = arr[y1:y2, x1:x2]
    if cropped.size == 0:
        return img
    return Image.fromarray(cropped)

def preprocess_pil(img: Image.Image, train: bool = False) -> Image.Image:
    img = img.convert("RGB")
    img = ImageOps.autocontrast(img)
    img = simple_hair_removal_pil(img)
    img = shades_of_gray_pil(img)
    img = rough_lesion_crop(img)
    return img

def build_transforms(train: bool = True):
    if train:
        return transforms.Compose([
            transforms.Lambda(lambda x: preprocess_pil(x, train=True)),
            transforms.RandomResizedCrop(CFG.IMG_SIZE, scale=(0.85, 1.0), ratio=(0.90, 1.10)),
            transforms.RandomHorizontalFlip(p=0.5),
            transforms.RandomVerticalFlip(p=0.5),
            transforms.RandomRotation(25),
            transforms.ColorJitter(brightness=0.08, contrast=0.08, saturation=0.08, hue=0.02),
            transforms.ToTensor(),
            transforms.Normalize(CFG.MEAN, CFG.STD),
        ])
    return transforms.Compose([
        transforms.Lambda(lambda x: preprocess_pil(x, train=False)),
        transforms.Resize((CFG.IMG_SIZE + 16, CFG.IMG_SIZE + 16)),
        transforms.CenterCrop(CFG.IMG_SIZE),
        transforms.ToTensor(),
        transforms.Normalize(CFG.MEAN, CFG.STD),
    ])

# =========================================================
# TABULAR PREPROCESSOR
# =========================================================
class TabularPreprocessor:
    def __init__(self, num_cols, cat_cols):
        self.num_cols = list(num_cols)
        self.cat_cols = list(cat_cols)
        self.num_means = {}
        self.num_stds = {}
        self.cat_maps = {}
        self.num_out_cols = []
        self.cat_out_cols = []

    def fit(self, df):
        for c in self.num_cols:
            x = pd.to_numeric(df[c], errors="coerce")
            med = float(x.median()) if x.notna().any() else 0.0
            std = float(x.std(ddof=0)) if x.notna().any() and float(x.std(ddof=0)) > 0 else 1.0
            self.num_means[c] = med
            self.num_stds[c] = std
            self.num_out_cols.append(c)
        for c in self.cat_cols:
            x = df[c].fillna("unknown").astype(str).str.lower().str.strip()
            uniq = ["unknown"] + sorted([u for u in x.unique().tolist() if u != "unknown"])
            self.cat_maps[c] = {v: i for i, v in enumerate(uniq)}
            self.cat_out_cols.append(c)
        return self

    def transform_row(self, row):
        nums = []
        for c in self.num_cols:
            v = row.get(c, np.nan)
            v = pd.to_numeric(v, errors="coerce")
            if pd.isna(v):
                v = self.num_means[c]
            v = (float(v) - self.num_means[c]) / self.num_stds[c]
            nums.append(v)
        cats = []
        for c in self.cat_cols:
            v = str(row.get(c, "unknown")).lower().strip()
            cats.append(self.cat_maps[c].get(v, 0))
        return np.array(nums, dtype=np.float32), np.array(cats, dtype=np.int64)

    @property
    def num_features(self):
        return len(self.num_cols)

    @property
    def cat_cardinalities(self):
        return [len(self.cat_maps[c]) for c in self.cat_cols]

# =========================================================
# DATASET
# =========================================================
def resolve_image_path(image_name, image_dirs):
    if pd.isna(image_name):
        return None
    base = canonical_image_name(image_name)
    candidates = []
    for d in image_dirs:
        candidates.extend([
            os.path.join(d, base),
            os.path.join(d, os.path.basename(base)),
            os.path.join(d, str(image_name)),
            os.path.join(d, canonical_image_name(str(image_name))),
        ])
    for p in candidates:
        if os.path.exists(p):
            return p
    # recursive fallback
    for d in image_dirs:
        for root, _, files in os.walk(d):
            for f in files:
                if f == os.path.basename(base):
                    return os.path.join(root, f)
    return None

class SkinLesionDataset(Dataset):
    def __init__(self, df, image_col, label_col, preprocessor, image_dirs, transform=None):
        self.df = df.reset_index(drop=True)
        self.image_col = image_col
        self.label_col = label_col
        self.preprocessor = preprocessor
        self.image_dirs = image_dirs
        self.transform = transform

        self._cache = {}
        paths = []
        keep_rows = []
        for i, row in self.df.iterrows():
            p = resolve_image_path(row[image_col], image_dirs)
            if p is not None:
                paths.append(p)
                keep_rows.append(i)
        self.df = self.df.iloc[keep_rows].reset_index(drop=True)
        self.paths = paths
        missing = len(df) - len(self.df)
        if missing:
            print(f"[WARN] {missing} images could not be resolved and were filtered out.")

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img_path = self.paths[idx]
        if img_path is None:
            raise FileNotFoundError(f"Could not resolve image path for row idx {idx}")
        img = Image.open(img_path).convert("RGB")
        if self.transform is not None:
            img = self.transform(img)
        x_num, x_cat = self.preprocessor.transform_row(row)
        y = encode_binary_label(row[self.label_col])
        return {
            "image": img,
            "x_num": torch.tensor(x_num, dtype=torch.float32),
            "x_cat": torch.tensor(x_cat, dtype=torch.long),
            "label": torch.tensor(y, dtype=torch.float32),
            "path": img_path,
        }

# =========================================================
# METRICS / CALIBRATION
# =========================================================
def compute_metrics(y_true, y_prob, threshold=0.5):
    y_true = np.asarray(y_true).astype(int)
    y_prob = np.asarray(y_prob)
    y_pred = (y_prob >= threshold).astype(int)

    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()

    out = {
        "accuracy": accuracy_score(y_true, y_pred),
        "precision": precision_score(y_true, y_pred, zero_division=0),
        "recall": recall_score(y_true, y_pred, zero_division=0),
        "f1": f1_score(y_true, y_pred, zero_division=0),
        "roc_auc": roc_auc_score(y_true, y_prob) if len(np.unique(y_true)) > 1 else float("nan"),
        "pr_auc": average_precision_score(y_true, y_prob) if len(np.unique(y_true)) > 1 else float("nan"),
        "sensitivity": tp / (tp + fn + 1e-9),
        "specificity": tn / (tn + fp + 1e-9),
        "ppv": tp / (tp + fp + 1e-9),
        "npv": tn / (tn + fn + 1e-9),
        "false_negative_rate": fn / (fn + tp + 1e-9),
        "youden_index": (tp / (tp + fn + 1e-9)) + (tn / (tn + fp + 1e-9)) - 1.0,
        "confusion_matrix": np.array([[tn, fp], [fn, tp]]),
    }
    return out

def sensitivity_at_specificity(y_true, y_prob, target_specificity=0.90):
    fpr, tpr, thr = roc_curve(y_true, y_prob)
    spec = 1 - fpr
    idx = int(np.argmin(np.abs(spec - target_specificity)))
    return float(tpr[idx]), float(thr[idx]), float(spec[idx])

class TemperatureScaler(nn.Module):
    def __init__(self):
        super().__init__()
        self.temperature = nn.Parameter(torch.ones(1) * 1.0)

    def forward(self, logits):
        return logits / self.temperature.clamp_min(1e-6)

    def fit(self, logits, labels):
        logits = torch.as_tensor(logits, dtype=torch.float32, device=device)
        labels = torch.as_tensor(labels, dtype=torch.float32, device=device)
        optimizer = torch.optim.LBFGS([self.temperature], lr=0.01, max_iter=50)

        def closure():
            optimizer.zero_grad()
            loss = F.binary_cross_entropy_with_logits(self.forward(logits).squeeze(-1), labels)
            loss.backward()
            return loss

        optimizer.step(closure)
        return self

def expected_calibration_error(y_true, y_prob, n_bins=15):
    y_true = np.asarray(y_true).astype(int)
    y_prob = np.asarray(y_prob)
    bins = np.linspace(0.0, 1.0, n_bins + 1)
    ece = 0.0
    for i in range(n_bins):
        lo, hi = bins[i], bins[i + 1]
        mask = (y_prob >= lo) & (y_prob < hi) if i < n_bins - 1 else (y_prob >= lo) & (y_prob <= hi)
        if mask.sum() == 0:
            continue
        acc = y_true[mask].mean()
        conf = y_prob[mask].mean()
        ece += (mask.mean()) * abs(acc - conf)
    return float(ece)

def reliability_curve_plot(y_true, y_prob, title="Reliability Diagram", n_bins=10):
    y_true = np.asarray(y_true).astype(int)
    y_prob = np.asarray(y_prob)
    bins = np.linspace(0, 1, n_bins + 1)
    bin_centers = (bins[:-1] + bins[1:]) / 2
    accs, confs = [], []
    for i in range(n_bins):
        lo, hi = bins[i], bins[i + 1]
        mask = (y_prob >= lo) & (y_prob < hi) if i < n_bins - 1 else (y_prob >= lo) & (y_prob <= hi)
        if mask.sum() == 0:
            accs.append(np.nan)
            confs.append(np.nan)
        else:
            accs.append(y_true[mask].mean())
            confs.append(y_prob[mask].mean())
    plt.figure(figsize=(6, 6))
    plt.plot([0, 1], [0, 1], linestyle="--")
    plt.plot(confs, accs, marker="o")
    plt.xlabel("Confidence")
    plt.ylabel("Observed accuracy")
    plt.title(title)
    plt.grid(True, alpha=0.25)
    plt.tight_layout()
    plt.show()

# =========================================================
# LOSS / TRAINING HELPERS
# =========================================================
class FocalLossWithLogits(nn.Module):
    def __init__(self, alpha=0.25, gamma=2.0, label_smoothing=0.10):
        super().__init__()
        self.alpha = alpha
        self.gamma = gamma
        self.label_smoothing = label_smoothing

    def forward(self, logits, targets):
        targets = targets.float()
        if self.label_smoothing > 0:
            targets = targets * (1 - self.label_smoothing) + 0.5 * self.label_smoothing
        bce = F.binary_cross_entropy_with_logits(logits, targets, reduction="none")
        prob = torch.sigmoid(logits)
        pt = targets * prob + (1 - targets) * (1 - prob)
        focal = self.alpha * (1 - pt).pow(self.gamma) * bce
        return focal.mean()

def build_optimizer(model):
    backbone_params, head_params = [], []
    for name, p in model.named_parameters():
        if not p.requires_grad:
            continue
        if any(k in name.lower() for k in ["backbone", "image_encoder", "stem", "down", "proj"]):
            backbone_params.append(p)
        else:
            head_params.append(p)
    return torch.optim.AdamW(
        [
            {"params": backbone_params, "lr": CFG.LR_BACKBONE},
            {"params": head_params, "lr": CFG.LR_HEAD},
        ],
        weight_decay=CFG.WEIGHT_DECAY,
    )

def freeze_initial_blocks(model, n=4):
    # Safe no-op for custom modules; for pretrained backbones freeze first blocks if present.
    for name, module in model.named_modules():
        if any(key in name.lower() for key in ["stem", "down1", "down2", "layers.0", "layers.1"]):
            for p in module.parameters(recurse=False):
                p.requires_grad = False

def cosine_with_warmup(step, total_steps, warmup_steps):
    if total_steps <= 0:
        return 1.0
    if step < warmup_steps:
        return float(step + 1) / float(max(1, warmup_steps))
    progress = float(step - warmup_steps) / float(max(1, total_steps - warmup_steps))
    return 0.5 * (1.0 + math.cos(math.pi * progress))

class EMA:
    def __init__(self, model, alpha=0.3):
        self.alpha = alpha
        self.shadow = {k: v.detach().clone() for k, v in model.state_dict().items() if v.dtype.is_floating_point}

    def update(self, model):
        for k, v in model.state_dict().items():
            if k in self.shadow and v.dtype.is_floating_point:
                self.shadow[k].mul_(1.0 - self.alpha).add_(v.detach(), alpha=self.alpha)

    def apply(self, model):
        self.backup = {}
        state = model.state_dict()
        for k, v in self.shadow.items():
            self.backup[k] = state[k].detach().clone()
            state[k].copy_(v)

    def restore(self, model):
        state = model.state_dict()
        for k, v in self.backup.items():
            state[k].copy_(v)
        self.backup = {}

def batch_to_device(batch):
    return {
        "image": batch["image"].to(device, non_blocking=True),
        "x_num": batch["x_num"].to(device, non_blocking=True),
        "x_cat": batch["x_cat"].to(device, non_blocking=True),
        "label": batch["label"].to(device, non_blocking=True),
        "path": batch["path"],
    }

def train_one_epoch(model, loader, optimizer, scaler, criterion, scheduler=None, ema=None):
    model.train()
    total_loss = 0.0
    all_preds, all_probs, all_targets = [], [], []
    optimizer.zero_grad(set_to_none=True)

    pbar = tqdm(loader, desc="train", leave=False)
    for step, batch in enumerate(pbar):
        batch = batch_to_device(batch)
        with autocast(enabled=(device.type == "cuda")):
            logits = model(batch["image"], batch["x_num"], batch["x_cat"])
            loss = criterion(logits, batch["label"])
            loss = loss / CFG.GRAD_ACCUM

        scaler.scale(loss).backward()

        if (step + 1) % CFG.GRAD_ACCUM == 0 or (step + 1) == len(loader):
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad(set_to_none=True)
            if scheduler is not None:
                scheduler.step()
            if ema is not None:
                ema.update(model)

        probs = torch.sigmoid(logits.detach()).float().cpu().numpy()
        preds = (probs >= 0.5).astype(int)
        targets = batch["label"].detach().cpu().numpy().astype(int)
        total_loss += loss.item() * CFG.GRAD_ACCUM
        all_probs.extend(probs.tolist())
        all_preds.extend(preds.tolist())
        all_targets.extend(targets.tolist())
        pbar.set_postfix(loss=float(loss.item() * CFG.GRAD_ACCUM))

    return total_loss / max(1, len(loader)), compute_metrics(all_targets, all_probs)

@torch.no_grad()
def predict_logits(model, loader):
    model.eval()
    logits_list, probs_list, targets_list, paths_list = [], [], [], []
    pbar = tqdm(loader, desc="eval", leave=False)
    for batch in pbar:
        batch = batch_to_device(batch)
        with autocast(enabled=(device.type == "cuda")):
            logits = model(batch["image"], batch["x_num"], batch["x_cat"])
        probs = torch.sigmoid(logits).float().cpu().numpy()
        logits_list.append(logits.detach().cpu().numpy())
        probs_list.append(probs)
        targets_list.append(batch["label"].detach().cpu().numpy())
        paths_list.extend(batch["path"])
    logits_arr = np.concatenate(logits_list).reshape(-1)
    probs_arr = np.concatenate(probs_list).reshape(-1)
    targets_arr = np.concatenate(targets_list).reshape(-1).astype(int)
    return logits_arr, probs_arr, targets_arr, paths_list

def evaluate_split(model, loader, threshold=0.5):
    logits, probs, y_true, paths = predict_logits(model, loader)
    metrics = compute_metrics(y_true, probs, threshold=threshold)
    metrics["ece"] = expected_calibration_error(y_true, probs)
    metrics["brier"] = brier_score_loss(y_true, probs)
    sens90, thr90, spec90 = sensitivity_at_specificity(y_true, probs, target_specificity=0.90)
    metrics["sensitivity_at_90spec"] = sens90
    metrics["threshold_at_90spec"] = thr90
    metrics["specificity_at_90spec"] = spec90
    return metrics, logits, probs, y_true, paths

def fit_threshold_by_youden(y_true, y_prob):
    fpr, tpr, thr = roc_curve(y_true, y_prob)
    j = tpr - fpr
    idx = int(np.argmax(j))
    return float(thr[idx]), float(j[idx])

# =========================================================
# XAI (ONE BEAUTIFUL FIGURE)
# =========================================================
def denormalize_tensor(img_t):
    mean = torch.tensor(CFG.MEAN, device=img_t.device).view(3, 1, 1)
    std = torch.tensor(CFG.STD, device=img_t.device).view(3, 1, 1)
    x = img_t * std + mean
    return torch.clamp(x, 0, 1)

def blurred_baseline(image):
    # image: [1,3,H,W] tensor normalized
    img = denormalize_tensor(image[0]).detach().cpu().permute(1, 2, 0).numpy()
    img_u8 = (img * 255).astype(np.uint8)
    blurred = cv2.GaussianBlur(img_u8, (31, 31), 0)
    baseline = torch.from_numpy(blurred.astype(np.float32) / 255.0).permute(2, 0, 1).unsqueeze(0)
    baseline = (baseline - torch.tensor(CFG.MEAN).view(1,3,1,1)) / torch.tensor(CFG.STD).view(1,3,1,1)
    return baseline.to(image.device)

def integrated_gradients_image(model, image, x_num, x_cat, target_index=1, steps=CFG.XAI_IG_STEPS):
    model.eval()
    baseline = blurred_baseline(image)
    total_grad = torch.zeros_like(image)

    for alpha in torch.linspace(0, 1, steps, device=image.device):
        x = baseline + alpha * (image - baseline)
        x.requires_grad_(True)
        logits = model(x, x_num, x_cat)
        prob = torch.sigmoid(logits)
        score = prob[:, target_index] if prob.ndim > 1 else prob
        if score.ndim == 0:
            score = score.unsqueeze(0)
        model.zero_grad(set_to_none=True)
        score.sum().backward(retain_graph=True)
        total_grad += x.grad.detach()

    avg_grad = total_grad / steps
    attr = (image - baseline) * avg_grad
    sal = attr.abs().sum(dim=1, keepdim=True)
    sal = sal / (sal.max(dim=-1, keepdim=True)[0].max(dim=-2, keepdim=True)[0] + 1e-9)
    return sal.detach()

def show_one_xai_figure(model, loader, title_suffix=""):
    batch = next(iter(loader))
    batch = batch_to_device(batch)
    idx = 0
    image = batch["image"][idx:idx+1]
    x_num = batch["x_num"][idx:idx+1]
    x_cat = batch["x_cat"][idx:idx+1]
    label = int(batch["label"][idx].item())

    model.eval()
    with torch.no_grad():
        logit = model(image, x_num, x_cat)
        prob = torch.sigmoid(logit).item()

    attr = integrated_gradients_image(model, image, x_num, x_cat, target_index=1)
    img_vis = denormalize_tensor(image[0]).permute(1,2,0).detach().cpu().numpy()
    heat = attr[0, 0].detach().cpu().numpy()

    plt.figure(figsize=(7.2, 7.2))
    plt.imshow(img_vis)
    plt.imshow(heat, cmap="jet", alpha=0.35)
    plt.axis("off")
    plt.title(f"{MODEL_NAME} | true={label} | melanoma prob={prob:.3f} {title_suffix}")
    plt.tight_layout()
    plt.show()

# =========================================================
# MAIN PIPELINE
# =========================================================
def load_splits():
    train_df = pd.read_csv(TRAIN_CSV)
    val_df = pd.read_csv(VAL_CSV)
    test_df = pd.read_csv(TEST_CSV)
    return train_df, val_df, test_df

def normalize_split_df(df):
    df = df.copy()
    image_col, label_col, num_cols, cat_cols = humanize_label_cols(df)
    if image_col != "image_fixed":
        df["image_fixed"] = df[image_col].apply(canonical_image_name)
        image_col = "image_fixed"
    df[label_col] = df[label_col].apply(encode_binary_label)
    return df, image_col, label_col, num_cols, cat_cols

train_df, val_df, test_df = load_splits()
train_df, image_col, label_col, num_cols, cat_cols = normalize_split_df(train_df)
val_df, _, _, _, _ = normalize_split_df(val_df)
test_df, _, _, _, _ = normalize_split_df(test_df)

# Keep only columns present across splits if possible
shared_num_cols = [c for c in num_cols if c in val_df.columns and c in test_df.columns]
shared_cat_cols = [c for c in cat_cols if c in val_df.columns and c in test_df.columns]
num_cols = shared_num_cols
cat_cols = shared_cat_cols

print("Image column :", image_col)
print("Label column :", label_col)
print("Numeric cols :", num_cols)
print("Categorical cols :", cat_cols)
print("Train / Val / Test:", len(train_df), len(val_df), len(test_df))
print(train_df[[label_col]].head())

preprocessor = TabularPreprocessor(num_cols, cat_cols).fit(train_df)

train_dirs = infer_image_dirs(TRAIN_CSV)
val_dirs   = infer_image_dirs(VAL_CSV)
test_dirs  = infer_image_dirs(TEST_CSV)

# fallback to root if needed
if not train_dirs:
    train_dirs = [os.path.join(DATA_ROOT, "train"), os.path.join(DATA_ROOT, "train", "images"), DATA_ROOT]
if not val_dirs:
    val_dirs = [os.path.join(DATA_ROOT, "val"), os.path.join(DATA_ROOT, "val", "images"), DATA_ROOT]
if not test_dirs:
    test_dirs = [os.path.join(DATA_ROOT, "test"), os.path.join(DATA_ROOT, "test", "images"), DATA_ROOT]

train_ds = SkinLesionDataset(train_df, image_col, label_col, preprocessor, train_dirs, transform=build_transforms(train=True))
val_ds   = SkinLesionDataset(val_df,   image_col, label_col, preprocessor, val_dirs,   transform=build_transforms(train=False))
test_ds  = SkinLesionDataset(test_df,  image_col, label_col, preprocessor, test_dirs,  transform=build_transforms(train=False))

# Weighted sampler for imbalance
train_labels = train_df[label_col].apply(encode_binary_label).values
class_counts = np.bincount(train_labels, minlength=2)
class_weights = 1.0 / np.maximum(class_counts, 1)
sample_weights = class_weights[train_labels]
sampler = WeightedRandomSampler(weights=torch.as_tensor(sample_weights, dtype=torch.double),
                                num_samples=len(sample_weights),
                                replacement=True)

train_loader = DataLoader(train_ds, batch_size=CFG.BATCH_SIZE, sampler=sampler,
                          num_workers=CFG.NUM_WORKERS, pin_memory=True, drop_last=True)
val_loader   = DataLoader(val_ds, batch_size=CFG.BATCH_SIZE, shuffle=False,
                          num_workers=CFG.NUM_WORKERS, pin_memory=True)
test_loader  = DataLoader(test_ds, batch_size=CFG.BATCH_SIZE, shuffle=False,
                          num_workers=CFG.NUM_WORKERS, pin_memory=True)

# Model build injected in the model-specific cell


In [ ]:

class GraphAttentionLayer(nn.Module):
    def __init__(self, d=128, dropout=0.15):
        super().__init__()
        self.W = nn.Linear(d, d, bias=False)
        self.a = nn.Linear(d * 2, 1, bias=False)
        self.leakyrelu = nn.LeakyReLU(0.2)
        self.dropout = nn.Dropout(dropout)
        self.norm = nn.LayerNorm(d)

    def forward(self, nodes):
        # nodes: [B, N, D]
        h = self.W(nodes)
        B, N, D = h.shape
        h_i = h.unsqueeze(2).expand(B, N, N, D)
        h_j = h.unsqueeze(1).expand(B, N, N, D)
        e = self.leakyrelu(self.a(torch.cat([h_i, h_j], dim=-1)).squeeze(-1))
        alpha = torch.softmax(e, dim=-1)
        alpha = self.dropout(alpha)
        out = torch.matmul(alpha, h)
        return self.norm(out + h)

class MetadataGraphEncoder(nn.Module):
    def __init__(self, num_numeric, cat_cardinalities, d=128, num_layers=2):
        super().__init__()
        self.num_numeric = num_numeric
        self.num_tokens = nn.ModuleList([nn.Linear(1, d) for _ in range(num_numeric)])
        self.cat_tokens = nn.ModuleList([nn.Embedding(card, d) for card in cat_cardinalities])
        self.layers = nn.ModuleList([GraphAttentionLayer(d=d) for _ in range(num_layers)])
        self.null_token = nn.Parameter(torch.zeros(1, 1, d))

    def forward(self, x_num, x_cat):
        tokens = []
        if x_num is not None and x_num.numel() > 0:
            for i, layer in enumerate(self.num_tokens):
                tokens.append(layer(x_num[:, i:i+1]).unsqueeze(1))
        if x_cat is not None and x_cat.numel() > 0:
            for i, emb in enumerate(self.cat_tokens):
                tokens.append(emb(x_cat[:, i]).unsqueeze(1))
        if len(tokens) == 0:
            return self.null_token.expand(x_num.size(0), 1, -1)
        nodes = torch.cat(tokens, dim=1)
        for layer in self.layers:
            nodes = layer(nodes)
        return nodes

class SwinV2GraphFusionClassifier(nn.Module):
    def __init__(self, num_numeric, cat_cardinalities, d=128, dropout=0.15):
        super().__init__()
        try:
            from torchvision.models import swin_v2_t, Swin_V2_T_Weights
            weights = Swin_V2_T_Weights.DEFAULT
            self.backbone = swin_v2_t(weights=weights)
        except Exception:
            from torchvision.models import swin_t, Swin_T_Weights
            weights = Swin_T_Weights.DEFAULT
            self.backbone = swin_t(weights=weights)
        self.backbone.head = nn.Identity()
        image_dim = getattr(self.backbone, "num_features", 768)
        self.img_proj = nn.Sequential(
            nn.Linear(image_dim, d),
            nn.GELU(),
            nn.Dropout(dropout),
        )
        self.meta_encoder = MetadataGraphEncoder(num_numeric, cat_cardinalities, d=d, num_layers=2)
        self.meta_pool = nn.Sequential(
            nn.Linear(d, d),
            nn.GELU(),
            nn.Dropout(dropout),
        )
        self.fuse = nn.Sequential(
            nn.Linear(d * 3, d * 2),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(d * 2, d),
            nn.GELU(),
            nn.Dropout(dropout),
        )
        self.cls = nn.Linear(d, 1)

    def forward(self, image, x_num, x_cat):
        img_feat = self.backbone(image)
        if img_feat.ndim > 2:
            img_feat = torch.flatten(img_feat, 1)
        img_feat = self.img_proj(img_feat)
        nodes = self.meta_encoder(x_num, x_cat)
        meta_feat = nodes.mean(dim=1)
        meta_feat = self.meta_pool(meta_feat)
        fused = self.fuse(torch.cat([img_feat, meta_feat, img_feat * meta_feat], dim=-1))
        return self.cls(fused).squeeze(-1)


In [ ]:

# =========================================================
# MODEL INSTANTIATION
# =========================================================
model = SwinV2GraphFusionClassifier(
    num_numeric=preprocessor.num_features,
    cat_cardinalities=preprocessor.cat_cardinalities,
    d=CFG.D,
    num_heads=CFG.NUM_HEADS,
    ff_dim=CFG.FF_DIM,
    num_layers=CFG.NUM_CA_LAYERS,
    dropout=CFG.DROPOUT_TF,
).to(device)

print(model.__class__.__name__)
total_train_steps = max(1, math.ceil(len(train_loader) / CFG.GRAD_ACCUM) * CFG.EPOCHS)
warmup_steps = max(1, CFG.WARMUP_EPOCHS * math.ceil(len(train_loader) / CFG.GRAD_ACCUM))
print("Train steps:", total_train_steps, "Warmup steps:", warmup_steps)

optimizer = build_optimizer(model)
scheduler = torch.optim.lr_scheduler.LambdaLR(
    optimizer,
    lr_lambda=lambda step: cosine_with_warmup(step, total_train_steps, warmup_steps),
)
scaler = GradScaler(enabled=(device.type == "cuda"))
criterion = FocalLossWithLogits(alpha=CFG.FOCAL_ALPHA, gamma=CFG.FOCAL_GAMMA, label_smoothing=CFG.LABEL_SMOOTH)
ema = EMA(model, alpha=CFG.EMA_ALPHA)

best_state = None
best_auc = -1.0
best_epoch = -1
patience = 0

# =========================================================
# TRAINING LOOP
# =========================================================
for epoch in range(CFG.EPOCHS):
    print(f"\nEpoch {epoch+1}/{CFG.EPOCHS}")
    train_loss, train_metrics = train_one_epoch(model, train_loader, optimizer, scaler, criterion, scheduler, ema)

    ema.apply(model)
    val_metrics, val_logits, val_probs, val_y, val_paths = evaluate_split(model, val_loader, threshold=0.5)
    ema.restore(model)

    print(
        f"train_loss={train_loss:.4f} | "
        f"train_auc={train_metrics['roc_auc']:.4f} | "
        f"val_auc={val_metrics['roc_auc']:.4f} | "
        f"val_f1={val_metrics['f1']:.4f} | "
        f"val_sens={val_metrics['sensitivity']:.4f} | "
        f"val_spec={val_metrics['specificity']:.4f}"
    )

    if val_metrics["roc_auc"] > best_auc:
        best_auc = val_metrics["roc_auc"]
        best_epoch = epoch + 1
        best_state = copy.deepcopy(model.state_dict())
        patience = 0
        print(f"  -> new best model saved (AUC={best_auc:.4f})")
    else:
        patience += 1
        print(f"  -> no improvement. patience={patience}/{CFG.PATIENCE}")
        if patience >= CFG.PATIENCE:
            print("Early stopping triggered.")
            break

# Load best checkpoint
if best_state is not None:
    model.load_state_dict(best_state)

print(f"Best epoch: {best_epoch}, best val AUC: {best_auc:.4f}")

# =========================================================
# VALIDATION CALIBRATION
# =========================================================
val_metrics, val_logits, val_probs, val_y, val_paths = evaluate_split(model, val_loader, threshold=0.5)
temp_scaler = TemperatureScaler().fit(val_logits, val_y)

test_logits, test_probs, test_y, test_paths = predict_logits(model, test_loader)
calibrated_test_probs = torch.sigmoid(
    temp_scaler(torch.tensor(test_logits, dtype=torch.float32, device=device)).detach()
).cpu().numpy()

# Threshold tuned on validation using Youden index
youden_thr, youden_idx = fit_threshold_by_youden(val_y, val_probs)
print(f"Youden threshold from val: {youden_thr:.4f} (J={youden_idx:.4f})")

# =========================================================
# FINAL TEST EVALUATION
# =========================================================
raw_test_metrics = compute_metrics(test_y, test_probs, threshold=youden_thr)
cal_test_metrics = compute_metrics(test_y, calibrated_test_probs, threshold=youden_thr)
cal_test_metrics["ece"] = expected_calibration_error(test_y, calibrated_test_probs)
cal_test_metrics["brier"] = brier_score_loss(test_y, calibrated_test_probs)
sens90, thr90, spec90 = sensitivity_at_specificity(test_y, calibrated_test_probs, target_specificity=0.90)
cal_test_metrics["sensitivity_at_90spec"] = sens90
cal_test_metrics["threshold_at_90spec"] = thr90
cal_test_metrics["specificity_at_90spec"] = spec90

print("\n=== Raw Test Metrics ===")
for k in ["accuracy","precision","recall","f1","roc_auc","pr_auc","sensitivity","specificity","ppv","npv","false_negative_rate","youden_index"]:
    print(f"{k:>24s}: {raw_test_metrics[k]:.4f}")

print("\n=== Calibrated Test Metrics ===")
for k in ["accuracy","precision","recall","f1","roc_auc","pr_auc","sensitivity","specificity","ppv","npv","false_negative_rate","youden_index","ece","brier","sensitivity_at_90spec"]:
    print(f"{k:>24s}: {cal_test_metrics[k]:.4f}")

print("\nConfusion matrix (calibrated, Youden threshold):")
print(cal_test_metrics["confusion_matrix"])

# Calibration plot
reliability_curve_plot(test_y, calibrated_test_probs, title=f"{MODEL_NAME} Reliability Diagram", n_bins=10)

# =========================================================
# UNCERTAINTY (MC DROPOUT)
# =========================================================
@torch.no_grad()
def mc_dropout_predict(model, loader, passes=5):
    model.train()  # keep dropout active
    preds = []
    for _ in range(passes):
        _, probs, y_true, _ = predict_logits(model, loader)
        preds.append(probs)
    model.eval()
    stack = np.stack(preds, axis=0)
    mean = stack.mean(axis=0)
    std = stack.std(axis=0)
    return mean, std, y_true

mc_mean, mc_std, mc_y = mc_dropout_predict(model, test_loader, passes=min(5, CFG.TTA_STEPS))
print(f"MC dropout mean std: {mc_std.mean():.4f}")

# =========================================================
# ERROR ANALYSIS
# =========================================================
def error_analysis(paths, y_true, y_prob, thr):
    y_pred = (y_prob >= thr).astype(int)
    df = pd.DataFrame({
        "path": paths,
        "y_true": y_true,
        "y_prob": y_prob,
        "y_pred": y_pred,
    })
    df["error_type"] = np.where((df.y_true == 1) & (df.y_pred == 0), "false_negative",
                         np.where((df.y_true == 0) & (df.y_pred == 1), "false_positive", "correct"))
    print("\nFalse negatives (highest confidence errors):")
    display(df[df.error_type == "false_negative"].sort_values("y_prob").head(5))
    print("\nFalse positives (highest confidence errors):")
    display(df[df.error_type == "false_positive"].sort_values("y_prob", ascending=False).head(5))
    return df

error_df = error_analysis(test_paths, test_y, calibrated_test_probs, youden_thr)
error_df.to_csv("/kaggle/working/error_analysis.csv", index=False)

# =========================================================
# ONE BEAUTIFUL XAI FIGURE
# =========================================================
show_one_xai_figure(model, val_loader, title_suffix="(IG overlay)")

# Save predictions and checkpoint
pred_df = pd.DataFrame({
    "path": test_paths,
    "y_true": test_y,
    "prob_raw": test_probs,
    "prob_calibrated": calibrated_test_probs,
})
pred_df.to_csv("/kaggle/working/test_predictions.csv", index=False)
torch.save({
    "model_name": MODEL_NAME,
    "model_state_dict": model.state_dict(),
    "preprocessor": {
        "num_cols": preprocessor.num_cols,
        "cat_cols": preprocessor.cat_cols,
        "num_means": preprocessor.num_means,
        "num_stds": preprocessor.num_stds,
        "cat_maps": preprocessor.cat_maps,
    },
    "threshold": youden_thr,
    "temperature": float(temp_scaler.temperature.detach().cpu().item()),
}, "/kaggle/working/best_model.pth")

print("\nDone. Artifacts saved to /kaggle/working/")
